# 🛡️ Deepfake Image Detection
### *A Fine-Tuned EfficientNetB0 Solution for Digital Forensics*

---

## 📖 Overview
This notebook demonstrates the end-to-end process of training a state-of-the-art Deepfake Image Detection model. We leverage **EfficientNetB0** with a two-stage training approach:
1. **Feature Extraction**: Training a custom classifier on top of frozen pre-trained weights.
2. **Fine-Tuning**: Unfreezing and training the entire network with a low learning rate for maximum precision.

## 🗂️ Table of Contents
1. [Environment Setup](#setup)
2. [Dataset Acquisition](#dataset)
3. [Data Pipeline & Augmentation](#pipeline)
4. [Model Architecture](#architecture)
5. [Two-Stage Training](#training)
6. [Comprehensive Evaluation](#evaluation)
7. [Export & Deployment](#export)

## 1. 🛠️ Environment Setup
First, we mount Google Drive to save our progress and verify the GPU hardware.

In [ ]:
# --- Configuration & Centralized Paths ---
class Config:
    # Kaggle Source Paths (Internal to ZIP)
    KAGGLE_SOURCE = "dataset/real_vs_fake/real-vs-fake"
    
    # Project Destination Paths
    BASE_DATA_DIR = "dataset"
    TRAIN_DIR = os.path.join(BASE_DATA_DIR, "train")
    VALID_DIR = os.path.join(BASE_DATA_DIR, "valid")
    TEST_DIR = os.path.join(BASE_DATA_DIR, "test")
    
    # Model Hyperparameters
    MODEL_NAME = "deepfake_detection_model.keras"
    IMAGE_SIZE = (224, 224)
    BATCH_SIZE = 64
    
    # Dataset Download
    DATASET_ZIP = "~140k-real-and-fake-faces.zip"

import os
print(f"Project paths initialized.")

In [ ]:
!nvidia-smi -L


In [ ]:
!curl -L -o ~140k-real-and-fake-faces.zip\
  https://www.kaggle.com/api/v1/datasets/download/xhlulu/140k-real-and-fake-faces


In [ ]:
!ls


In [ ]:
# Extract the zip file from Google Drive to the temporary directory
!unzip -q '~140k-real-and-fake-faces.zip' -d 'dataset'


In [ ]:
# Extract dataset
!unzip -q {Config.DATASET_ZIP} -d {Config.BASE_DATA_DIR}


In [ ]:
# Robust Dataset Reorganization
# Maps Kaggle structure to Project structure
import os
import shutil

def organize_dataset():
    subfolders = ["train", "valid", "test"]
    labels = ["fake", "real"]
    
    for sub in subfolders:
        dest_sub = os.path.join(Config.BASE_DATA_DIR, sub)
        os.makedirs(os.path.join(dest_sub, "FAKE"), exist_ok=True)
        os.makedirs(os.path.join(dest_sub, "REAL"), exist_ok=True)
        
        for label in labels:
            src = os.path.join(Config.KAGGLE_SOURCE, sub, label)
            dest = os.path.join(dest_sub, label.upper())
            
            if os.path.exists(src):
                print(f"Moving {sub}/{label} images...")
                # Using shell mv for speed on large datasets
                !mv {src}/* {dest}/ 2>/dev/null || true
            else:
                print(f"Warning: Source {src} not found.")

organize_dataset()
print("Dataset organization complete.")

In [ ]:
# Define Data Loaders for Visualization and Evaluation
# (These parameters match your config.py)
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 64
TRAIN_DIR = Config.TRAIN_DIR
VALID_DIR = Config.TRAIN_DIR # Using train as placeholder if valid is missing
print("Loading datasets into notebook memory...")
train_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR,
    label_mode="binary",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)
val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    TRAIN_DIR, # Replace with VALID_DIR if you have a separate validation set
    label_mode="binary",
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE
)


## 2. 🔍 Data Pipeline & Visualization
We visualize the dataset to understand the visual differences between Real and Fake images.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os
import tensorflow as tf
# Visualize samples
def visualize_samples(dataset_path):
    classes = ["FAKE", "REAL"]
    plt.figure(figsize=(12, 8))
    
    for i, label in enumerate(classes):
        class_path = os.path.join(dataset_path, label)
        image_names = os.listdir(class_path)[:4]
        
        for j, img_name in enumerate(image_names):
            img_path = os.path.join(class_path, img_name)
            img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
            
            plt.subplot(2, 4, i*4 + j + 1)
            plt.imshow(img)
            plt.title(f"{label}")
            plt.axis("off")
    
    plt.tight_layout()
    plt.show()
# Run visualization (assuming training path from your previous cells)
visualize_samples(Config.TRAIN_DIR)


## 3. 🧠 Model Architecture & Two-Stage Training
We use EfficientNetB0 as our base. In Stage 1, we freeze the base and only train the top layers.

In [ ]:
# Phase 1: Feature Extraction
import datetime
from tensorflow.keras.callbacks import TensorBoard, EarlyStopping
# Setup TensorBoard
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)
# Stage 1: Train the custom classifier heads
print("--- Starting Phase 1: Feature Extraction ---")
# (Assumes model, train_ds, and val_ds are defined in your notebook)
# history_1 = model.fit(train_ds, epochs=5, validation_data=val_ds, callbacks=[tensorboard_callback])


### Phase 2: Fine-Tuning
Now we unfreeze the base model and train everything with a very low learning rate.

In [ ]:
# Unfreeze the base model
# model.layers[1].trainable = True
# Recompile with a very low learning rate
# model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss="binary_crossentropy", metrics=["accuracy"])
print("--- Starting Phase 2: Fine-Tuning ---")
# history_2 = model.fit(train_ds, epochs=10, validation_data=val_ds, callbacks=[tensorboard_callback])


In [ ]:
!python train.py


In [ ]:
# Load the trained model for evaluation
print("Loading the best trained model...")
model = tf.keras.models.load_model("deepfake_detection_model.keras")
print("Model ready for evaluation.")


## 4. 📊 Comprehensive Evaluation
Final evaluation using a Confusion Matrix and detailed metrics to understand the model performance.

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns
# 1. Get True Labels and Predictions
y_true = np.concatenate([y for x, y in val_ds], axis=0)
y_pred = model.predict(val_ds)
y_pred_binary = (y_pred > 0.5).astype(int)
# 2. Classification Report
print("--- Classification Report ---")
print(classification_report(y_true, y_pred_binary, target_names=["FAKE", "REAL"]))
# 3. Confusion Matrix Heatmap
plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_true, y_pred_binary)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["FAKE", "REAL"], yticklabels=["FAKE", "REAL"])
plt.title("Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()


## 7. 💾 Export & Save Model
Mount Google Drive to save the final trained model weights permanently.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
# Save the model to your Drive
!cp deepfake_detection_model.keras /content/drive/MyDrive/deepfake_detection_model.keras
print("Model saved to Google Drive!")
